# 03. Feature Engineering
----

This notebook transforms sentiment outputs into forecasting features.

Objectives:
- Generate sentiment labels for all news articles
- Aggregate sentiment into daily features
- Prepare a modeling-ready dataset

## 1. Load Data

In [1]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

import pandas as pd

from src import (
data_io as dio,
data_validation as dv,
sentiment_utils as su
)

In [2]:
DATA_PATH_NEWS = "../data/interim/economy_news_sentiment_input.parquet"
economy_sentiment_input = dio.load_dataframe(
    DATA_PATH_NEWS
)

economy_sentiment_input.shape

(6867, 14)

In [3]:
DATA_PATH_SENTIMENT_LABEL = "../data/interim/manual_sentiment_evaluation.parquet"
df_final = dio.load_dataframe(
    DATA_PATH_SENTIMENT_LABEL
)

df_final.head()

,Title,Label,predicted_label,match
0,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar ...",Positive,Positive,True
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Mil...",Negative,Neutral,False
2,"AS Pening, 'Badai' Baru Bikin Penampakan Reses...",Negative,Negative,True
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sam...",Positive,Neutral,False
4,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Neutral,Neutral,True


In [4]:
# Validation
economy_sentiment_input.info()

<class 'pandas.DataFrame'>
RangeIndex: 6867 entries, 0 to 6866
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id                  6867 non-null   int64         
 1   source              6867 non-null   str           
 2   title               6867 non-null   str           
 3   image               6858 non-null   str           
 4   url                 6867 non-null   str           
 5   content             6867 non-null   str           
 6   date                6867 non-null   datetime64[ms]
 7   embedding           6867 non-null   str           
 8   created_at          6867 non-null   str           
 9   updated_at          6867 non-null   str           
 10  summary             6866 non-null   str           
 11  nomic_embedding     6867 non-null   str           
 12  label               6867 non-null   str           
 13  text_for_sentiment  6867 non-null   str           
dtypes: 

## 2. Generate Sentiment for Entire Dataset

### 2.1. Generate Label

In [5]:
economy_sentiment_input["sentiment_label"] = (
    economy_sentiment_input["text_for_sentiment"]
    .apply(su.assign_sentiment_label)
)

### 2.1. Generate Score

In [6]:
economy_sentiment_input["sentiment_score"] = (
    economy_sentiment_input["sentiment_label"]
    .apply(su.convert_sentiment_to_score)
)

In [7]:
economy_sentiment_input[
    [
        "sentiment_label",
        "sentiment_score"
    ]
].head()

,sentiment_label,sentiment_score
0,Negative,-1
1,Neutral,0
2,Positive,1
3,Neutral,0
4,Neutral,0


In [8]:
economy_sentiment_input["sentiment_score"].value_counts()

sentiment_score
 0    4024
 1    1995
-1     848
Name: count, dtype: int64

### Sentiment Distribution

Analyze the distribution of sentiment labels generated by the baseline rule-based model.

In [9]:
economy_sentiment_input[
    "sentiment_label"
].value_counts(
    normalize=True
) * 100

sentiment_label
Neutral     58.599097
Positive    29.051988
Negative    12.348915
Name: proportion, dtype: float64

### Observation

The rule-based sentiment model predicts Neutral sentiment for the majority of news articles.

Possible reasons:
- The keyword dictionary is intentionally small.
- Many economic articles contain informative rather than emotional language.
- The model cannot capture semantic meaning beyond explicit keywords.

This behavior is expected from a baseline keyword-based approach.

In [10]:
dio.save_dataframe(
    economy_sentiment_input,
    "../data/interim/news_with_sentiment.parquet"
)

## 3. Daily Sentiment Aggregation

In [11]:
economy_sentiment_input[
    [
        "date",
        "sentiment_label",
        "sentiment_score"
    ]
].head()

,date,sentiment_label,sentiment_score
0,2023-03-16 07:16:28,Negative,-1
1,2023-03-19 02:11:48,Neutral,0
2,2023-03-22 05:45:35,Positive,1
3,2023-03-14 09:18:39,Neutral,0
4,2023-03-14 11:13:41,Neutral,0


In [12]:
# Validation
economy_sentiment_input["date"].dtype

dtype('<M8[ms]')

### 3.1. Generate Date Only Column

In [13]:
economy_sentiment_input["news_date"] = (
    economy_sentiment_input["date"]
    .dt.date
)

### 3.2. Average Sentiment

In [14]:
daily_sentiment = (
    economy_sentiment_input
    .groupby("news_date")
    ["sentiment_score"]
    .mean()
    .reset_index()
)

In [15]:
daily_sentiment = daily_sentiment.rename(
    columns={
        "sentiment_score":
        "avg_sentiment"
    }
)

In [16]:
daily_sentiment.head()

,news_date,avg_sentiment
0,2023-03-03,0.00
1,2023-03-04,0.50
2,2023-03-05,0.25
3,2023-03-06,0.20
4,2023-03-08,0.00


### 3.3. News Volume 

In [17]:
daily_news_count = (
    economy_sentiment_input
    .groupby("news_date")
    .size()
    .reset_index(
        name="news_count"
    )
)

In [18]:
daily_news_count.head()

,news_date,news_count
0,2023-03-03,1
1,2023-03-04,2
2,2023-03-05,4
3,2023-03-06,10
4,2023-03-08,5


### 3.4. Merge

In [19]:
daily_sentiment_df = (
    daily_sentiment
    .merge(
        daily_news_count,
        on="news_date",
        how="left"
    )
)

In [20]:
daily_sentiment_df.head()

,news_date,avg_sentiment,news_count
0,2023-03-03,0.00,1
1,2023-03-04,0.50,2
2,2023-03-05,0.25,4
3,2023-03-06,0.20,10
4,2023-03-08,0.00,5


In [21]:
# Save data
dio.save_dataframe(
    daily_sentiment_df,
    "../data/interim/daily_sentiment_aggregation.parquet"
)

In [22]:
daily_sentiment_df.describe()

,avg_sentiment,news_count
count,39.000000,39.000000
mean,0.172825,176.076923
std,0.094849,119.937568
min,0.000000,1.000000
25%,0.115166,81.000000
50%,0.149206,147.000000
75%,0.223964,285.500000
max,0.500000,409.000000


In [23]:
daily_sentiment_df["news_count"].describe()

count     39.000000
mean     176.076923
std      119.937568
min        1.000000
25%       81.000000
50%      147.000000
75%      285.500000
max      409.000000
Name: news_count, dtype: float64

## 4. Sentiment Composition Features

### 4.1. Count Sentiment Per Day

In [24]:
daily_sentiment_count = (
    economy_sentiment_input
    .groupby(
        [
            "news_date",
            "sentiment_label"
        ]
    )
    .size()
    .unstack(fill_value=0)
)

In [25]:
daily_sentiment_count.head()

sentiment_label,Negative,Neutral,Positive
news_date,,,
2023-03-03,0,1,0
2023-03-04,0,1,1
2023-03-05,0,3,1
2023-03-06,0,8,2
2023-03-08,1,3,1


### 4.2. Reset Index

In [26]:
daily_sentiment_count = (
    daily_sentiment_count
    .reset_index()
)

### 4.3. Calculate Ratios

In [27]:
daily_sentiment_count["total_news"] = (
    daily_sentiment_count[
        [
            "Positive",
            "Neutral",
            "Negative"
        ]
    ]
    .sum(axis=1)
)

#### Positive Ratio

In [28]:
daily_sentiment_count["positive_ratio"] = (
    daily_sentiment_count["Positive"]
    /
    daily_sentiment_count["total_news"]
)

#### Neutral Ratio

In [29]:
daily_sentiment_count["neutral_ratio"] = (
    daily_sentiment_count["Neutral"]
    /
    daily_sentiment_count["total_news"]
)

#### Negative Ratio

In [30]:
daily_sentiment_count["negative_ratio"] = (
    daily_sentiment_count["Negative"]
    /
    daily_sentiment_count["total_news"]
)

### 4.4. Keep Only Needed Columns

In [31]:
daily_sentiment_ratio = (
    daily_sentiment_count[
        [
            "news_date",
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio"
        ]
    ]
)

In [32]:
daily_sentiment_ratio.head()

sentiment_label,news_date,positive_ratio,neutral_ratio,negative_ratio
0,2023-03-03,0.00,1.00,0.0
1,2023-03-04,0.50,0.50,0.0
2,2023-03-05,0.25,0.75,0.0
3,2023-03-06,0.20,0.80,0.0
4,2023-03-08,0.20,0.60,0.2


In [33]:
(
    daily_sentiment_ratio[
        [
            "positive_ratio",
            "neutral_ratio",
            "negative_ratio"
        ]
    ]
    .sum(axis=1)
).head()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64

### 4.5. Merge `daily_sentiment_df` with `daily_sentiment_ratio`

In [34]:
daily_sentiment_features = (
    daily_sentiment_df
    .merge(
        daily_sentiment_ratio,
        on="news_date",
        how="left"
    )
)

In [35]:
daily_sentiment_features.head()

,news_date,avg_sentiment,news_count,positive_ratio,neutral_ratio,negative_ratio
0,2023-03-03,0.00,1,0.00,1.00,0.0
1,2023-03-04,0.50,2,0.50,0.50,0.0
2,2023-03-05,0.25,4,0.25,0.75,0.0
3,2023-03-06,0.20,10,0.20,0.80,0.0
4,2023-03-08,0.00,5,0.20,0.60,0.2


In [36]:
dio.save_dataframe(
    daily_sentiment_features,
    "../data/interim/daily_sentiment_features.parquet"
)

In [37]:
# Validation
daily_sentiment_features[
    [
        "positive_ratio",
        "neutral_ratio",
        "negative_ratio"
    ]
].sum(axis=1).describe()

count    3.900000e+01
mean     1.000000e+00
std      4.411579e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

## 5. Rolling Sentiment Features

Create rolling sentiment indicators to capture short-term sentiment trends.

Features:
- 3-day rolling sentiment
- 7-day rolling sentiment

### 5.1. Sort Date

In [38]:
daily_sentiment_features = (
    daily_sentiment_features
    .sort_values("news_date")
    .reset_index(drop=True)
)

### 5.2. 3-Day Rolling Mean

In [39]:
daily_sentiment_features["sentiment_ma_3"] = (
    daily_sentiment_features[
        "avg_sentiment"
    ]
    .rolling(
        window=3,
        min_periods=1
    )
    .mean()
)

### 5.3. 7-Day Rolling Mean

In [40]:
daily_sentiment_features["sentiment_ma_7"] = (
    daily_sentiment_features[
        "avg_sentiment"
    ]
    .rolling(
        window=7,
        min_periods=1
    )
    .mean()
)

In [41]:
# Validation
daily_sentiment_features[
    [
        "news_date",
        "avg_sentiment",
        "sentiment_ma_3",
        "sentiment_ma_7"
    ]
].head(10)

,news_date,avg_sentiment,sentiment_ma_3,sentiment_ma_7
0,2023-03-03,0.000000,0.000000,0.000000
1,2023-03-04,0.500000,0.250000,0.250000
2,2023-03-05,0.250000,0.250000,0.250000
3,2023-03-06,0.200000,0.316667,0.237500
4,2023-03-08,0.000000,0.150000,0.190000
5,2023-03-09,0.367347,0.189116,0.219558
6,2023-03-10,0.147059,0.171469,0.209201
7,2023-03-11,0.097561,0.203989,0.223138
8,2023-03-12,0.131148,0.125256,0.170445
9,2023-03-13,0.265700,0.164803,0.172688


#### Insights

Financial markets often react to sentiment trends rather than isolated news events. Therefore, rolling sentiment indicators were created to capture short-term market mood persistence.

In [42]:
# Save file

dio.save_dataframe(
    daily_sentiment_features,
    "../data/interim/daily_sentiment_features.parquet"
)

## 6. Lag Features

Create lagged sentiment variables to capture delayed market reactions.

Features:
- 1-day lag sentiment
- 3-day lag sentiment
- 7-day lag sentiment

### 6.1. Lag 1

In [43]:
daily_sentiment_features[
    "sentiment_lag_1"
] = (
    daily_sentiment_features[
        "avg_sentiment"
    ]
    .shift(1)
)

### 6.2. Lag 3

In [44]:
daily_sentiment_features[
    "sentiment_lag_3"
] = (
    daily_sentiment_features[
        "avg_sentiment"
    ]
    .shift(3)
)

### 6.3. Lag 7

In [45]:
daily_sentiment_features[
    "sentiment_lag_7"
] = (
    daily_sentiment_features[
        "avg_sentiment"
    ]
    .shift(7)
)

In [46]:
# Validation
daily_sentiment_features[
    [
        "news_date",
        "avg_sentiment",
        "sentiment_lag_1",
        "sentiment_lag_3",
        "sentiment_lag_7"
    ]
].head(10)

,news_date,avg_sentiment,sentiment_lag_1,sentiment_lag_3,sentiment_lag_7
0,2023-03-03,0.000000,NaN,NaN,NaN
1,2023-03-04,0.500000,0.000000,NaN,NaN
2,2023-03-05,0.250000,0.500000,NaN,NaN
3,2023-03-06,0.200000,0.250000,0.000000,NaN
4,2023-03-08,0.000000,0.200000,0.500000,NaN
5,2023-03-09,0.367347,0.000000,0.250000,NaN
6,2023-03-10,0.147059,0.367347,0.200000,NaN
7,2023-03-11,0.097561,0.147059,0.000000,0.00
8,2023-03-12,0.131148,0.097561,0.367347,0.50
9,2023-03-13,0.265700,0.131148,0.147059,0.25


In [47]:
# Save file

dio.save_dataframe(
    daily_sentiment_features,
    "../data/interim/daily_sentiment_features.parquet"
)

## 7. Load Market Data (IHSG)

In [48]:
economy_sentiment_input["date"].agg(
    ["min", "max"]
)

min   2023-03-03 13:00:00
max   2023-04-11 15:46:34
Name: date, dtype: datetime64[ms]

### 7.1. Market Target Selection

Selected Market Target:
- Jakarta Composite Index (IHSG)

Ticker:
- ^JKSE

Reason:
- Economic news affects the overall Indonesian market.
- The news dataset contains macroeconomic information rather than company-specific events.
- IHSG serves as a broad indicator of market sentiment and economic conditions.

### 7.2. Import Library

In [49]:
import yfinance as yf

### 7.3. Define Period

In [50]:
start_date = (
    economy_sentiment_input["date"]
    .min()
    .date()
)

end_date = (
    economy_sentiment_input["date"]
    .max()
    .date()
)

print(start_date)
print(end_date)

2023-03-03
2023-04-11


### 7.4. Download IHSG Data

In [51]:
# add buffer several days because of lag and rolling window
market_df = yf.download(
    "^JKSE",
    start="2023-02-20",
    end="2023-04-20",
    auto_adjust=True
)

[*********************100%***********************]  1 of 1 completed


### 7.5. Validation

In [52]:
market_df.head()

Price,Close,High,Low,Open,Volume
Ticker,^JKSE,^JKSE,^JKSE,^JKSE,^JKSE
Date,,,,,
2023-02-20,6894.716797,6910.504883,6863.670898,6895.855957,175859700
2023-02-21,6873.404785,6923.672852,6870.911133,6894.716797,144215700
2023-02-22,6809.967773,6875.390137,6781.229004,6873.263184,149215500
2023-02-23,6839.454102,6839.454102,6806.997070,6810.108887,124488600
2023-02-24,6856.576172,6880.312012,6839.454102,6839.454102,130300200


In [53]:
market_df.shape

(39, 5)

In [54]:
market_df.isnull().sum()

Price   Ticker
Close   ^JKSE     0
High    ^JKSE     0
Low     ^JKSE     0
Open    ^JKSE     0
Volume  ^JKSE     0
dtype: int64

### 7.6. Reset Index

In [55]:
market_df = (
    market_df
    .reset_index()
)

### 7.7. Rename Columns 

In [56]:
if isinstance(
    market_df.columns,
    pd.MultiIndex
):
    market_df.columns = [
        col[0].lower()
        for col in market_df.columns
    ]

In [57]:
market_df = (
    market_df
    .reset_index()
)

In [58]:
market_df.head()

,index,date,close,high,low,open,volume
0,0,2023-02-20,6894.716797,6910.504883,6863.670898,6895.855957,175859700
1,1,2023-02-21,6873.404785,6923.672852,6870.911133,6894.716797,144215700
2,2,2023-02-22,6809.967773,6875.390137,6781.229004,6873.263184,149215500
3,3,2023-02-23,6839.454102,6839.454102,6806.997070,6810.108887,124488600
4,4,2023-02-24,6856.576172,6880.312012,6839.454102,6839.454102,130300200


### 7.8. Keep Relevant Columns 

In [59]:
market_df = market_df[
    [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

In [60]:
# Validation
market_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   date    39 non-null     datetime64[s]
 1   open    39 non-null     float64      
 2   high    39 non-null     float64      
 3   low     39 non-null     float64      
 4   close   39 non-null     float64      
 5   volume  39 non-null     int64        
dtypes: datetime64[s](1), float64(4), int64(1)
memory usage: 2.0 KB


In [61]:
market_df.head()

,date,open,high,low,close,volume
0,2023-02-20,6895.855957,6910.504883,6863.670898,6894.716797,175859700
1,2023-02-21,6894.716797,6923.672852,6870.911133,6873.404785,144215700
2,2023-02-22,6873.263184,6875.390137,6781.229004,6809.967773,149215500
3,2023-02-23,6810.108887,6839.454102,6806.997070,6839.454102,124488600
4,2023-02-24,6839.454102,6880.312012,6839.454102,6856.576172,130300200


In [62]:
# Save data
dio.save_dataframe(
    market_df,
    "../data/interim/ihsg_market_data.parquet"
)

## 8. Technical Indicators

Create market-based features from IHSG historical prices.

Objectives:
- Capture market trend
- Capture short-term momentum
- Capture market volatility

Generated Features:
- Daily Return
- SMA 5
- SMA 10
- Volatility 5

### 8.1. Daily Return

In [63]:
market_df["daily_return"] = (
    market_df["close"]
    .pct_change()
)

In [64]:
# Validation
market_df[
    [
        "date",
        "close",
        "daily_return"
    ]
].head()

,date,close,daily_return
0,2023-02-20,6894.716797,NaN
1,2023-02-21,6873.404785,-0.003091
2,2023-02-22,6809.967773,-0.009229
3,2023-02-23,6839.454102,0.004330
4,2023-02-24,6856.576172,0.002503


### 8.2. SMA 5 (Simple Moving Average for 5 Days)

In [65]:
market_df["sma_5"] = (
    market_df["close"]
    .rolling(
        window=5,
        min_periods=1
    )
    .mean()
)

### 8.3. SMA 10 (Simple Moving  for 10 Days)

In [66]:
market_df["sma_10"] = (
    market_df["close"]
    .rolling(
        window=10,
        min_periods=1
    )
    .mean()
)

In [67]:
# Validation
market_df[
    [
        "date",
        "close",
        "sma_5",
        "sma_10"
    ]
].head(15)

,date,close,sma_5,sma_10
0,2023-02-20,6894.716797,6894.716797,6894.716797
1,2023-02-21,6873.404785,6884.060791,6884.060791
2,2023-02-22,6809.967773,6859.363118,6859.363118
3,2023-02-23,6839.454102,6854.385864,6854.385864
4,2023-02-24,6856.576172,6854.823926,6854.823926
5,2023-02-27,6854.776855,6846.835938,6854.816081
6,2023-02-28,6843.238770,6840.802734,6853.162179
7,2023-03-01,6844.936035,6847.796387,6852.133911
8,2023-03-02,6857.415039,6851.388574,6852.720703
9,2023-03-03,6813.636230,6842.800586,6848.812256


### 8.4. Volatility 5

How greatly a stock price swings around the mean price.

In [68]:
market_df["volatility_5"] = (
    market_df["daily_return"]
    .rolling(
        window=5,
        min_periods=1
    )
    .std()
)

In [69]:
# Validation
market_df[
    [
        "date",
        "daily_return",
        "volatility_5"
    ]
].head(10)

,date,daily_return,volatility_5
0,2023-02-20,NaN,NaN
1,2023-02-21,-0.003091,NaN
2,2023-02-22,-0.009229,0.004340
3,2023-02-23,0.004330,0.006790
4,2023-02-24,0.002503,0.006116
5,2023-02-27,-0.000262,0.005320
6,2023-02-28,-0.001683,0.005228
7,2023-03-01,0.000248,0.002382
8,2023-03-02,0.001823,0.001671
9,2023-03-03,-0.006384,0.003131


### 8.5. Missing Values Check

In [70]:
market_df.isnull().sum()

date            0
open            0
high            0
low             0
close           0
volume          0
daily_return    1
sma_5           0
sma_10          0
volatility_5    2
dtype: int64

In [71]:
# Save data
dio.save_dataframe(
    market_df,
    "../data/interim/ihsg_features.parquet"
)

## 9. Merge Sentiment Features with Market Features

Combine sentiment-derived features with market-derived features.

Challenges:
- News exists every day.
- Market only exists on trading days.

Solution:
- Merge sentiment information into trading days only.

### 9.1. Load Shape

In [72]:
daily_sentiment_features.shape

(39, 11)

In [73]:
market_df.shape

(39, 10)

### 9.2. Check Date Type

In [74]:
daily_sentiment_features["news_date"].dtype

dtype('O')

In [75]:
market_df["date"].dtype

dtype('<M8[s]')

In [76]:
# Casting Sentiment date into datetime

daily_sentiment_features["news_date"] = (
    pd.to_datetime(
        daily_sentiment_features["news_date"]
    )
)

In [77]:
# Casting Market date into datetime

market_df["date"] = (
    pd.to_datetime(
        market_df["date"]
    )
)

In [78]:
# Validation
daily_sentiment_features["news_date"].dtype

dtype('<M8[s]')

In [79]:
market_df["date"].dtype

dtype('<M8[s]')

### 9.3. Merge

- Market data as base table
- Target variable is from market data
- `Laft Join` because all trading day must be keep

In [80]:
forecasting_df = (
    market_df
    .merge(
        daily_sentiment_features,
        left_on="date",
        right_on="news_date",
        how="left"
    )
)

In [81]:
# Validation
forecasting_df.shape

(39, 21)

In [82]:
forecasting_df.head()

,date,open,high,low,close,volume,daily_return,sma_5,sma_10,volatility_5,...,avg_sentiment,news_count,positive_ratio,neutral_ratio,negative_ratio,sentiment_ma_3,sentiment_ma_7,sentiment_lag_1,sentiment_lag_3,sentiment_lag_7
0,2023-02-20,6895.855957,6910.504883,6863.670898,6894.716797,175859700,NaN,6894.716797,6894.716797,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-02-21,6894.716797,6923.672852,6870.911133,6873.404785,144215700,-0.003091,6884.060791,6884.060791,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-02-22,6873.263184,6875.390137,6781.229004,6809.967773,149215500,-0.009229,6859.363118,6859.363118,0.004340,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-02-23,6810.108887,6839.454102,6806.997070,6839.454102,124488600,0.004330,6854.385864,6854.385864,0.006790,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-02-24,6839.454102,6880.312012,6839.454102,6856.576172,130300200,0.002503,6854.823926,6854.823926,0.006116,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 9.4. Missing Values Check

In [83]:
forecasting_df.isnull().sum()

date                0
open                0
high                0
low                 0
close               0
volume              0
daily_return        1
sma_5               0
sma_10              0
volatility_5        2
news_date          15
avg_sentiment      15
news_count         15
positive_ratio     15
neutral_ratio      15
negative_ratio     15
sentiment_ma_3     15
sentiment_ma_7     15
sentiment_lag_1    16
sentiment_lag_3    16
sentiment_lag_7    20
dtype: int64

### 9.5. Fill Missing Sentiment 

In [84]:
sentiment_cols = [
    "avg_sentiment",
    "news_count",
    "positive_ratio",
    "neutral_ratio",
    "negative_ratio",
    "sentiment_ma_3",
    "sentiment_ma_7",
    "sentiment_lag_1",
    "sentiment_lag_3",
    "sentiment_lag_7"
]

In [85]:
forecasting_df[
    sentiment_cols
] = forecasting_df[
    sentiment_cols
].fillna(0)

In [86]:
# Validation
forecasting_df[
    sentiment_cols
].isnull().sum()

avg_sentiment      0
news_count         0
positive_ratio     0
neutral_ratio      0
negative_ratio     0
sentiment_ma_3     0
sentiment_ma_7     0
sentiment_lag_1    0
sentiment_lag_3    0
sentiment_lag_7    0
dtype: int64

### 9.6. Save Data

In [87]:
dio.save_dataframe(
    forecasting_df,
    "../data/interim/forecasting_base.parquet"
)

## 10. Create Forecasting Target

Create a binary target representing next-day market direction.

Target:
- 1 = Market goes up tomorrow
- 0 = Market goes down or remains unchanged tomorrow

### 10.1. Load Shape

In [88]:
forecasting_df["next_day_close"] = (
    forecasting_df[
        "close"
    ].shift(-1)
)

In [89]:
# Validation
forecasting_df[
    [
        "date",
        "close",
        "next_day_close"
    ]
].head()

,date,close,next_day_close
0,2023-02-20,6894.716797,6873.404785
1,2023-02-21,6873.404785,6809.967773
2,2023-02-22,6809.967773,6839.454102
3,2023-02-23,6839.454102,6856.576172
4,2023-02-24,6856.576172,6854.776855


### 10.2. Create Target

In [90]:
forecasting_df["target_direction"] = (
    forecasting_df["next_day_close"]
    >
    forecasting_df["close"]
).astype(int)

In [91]:
# Validation
forecasting_df[
    [
        "close",
        "next_day_close",
        "target_direction"
    ]
].head(10)

,close,next_day_close,target_direction
0,6894.716797,6873.404785,0
1,6873.404785,6809.967773,0
2,6809.967773,6839.454102,1
3,6839.454102,6856.576172,1
4,6856.576172,6854.776855,0
5,6854.776855,6843.238770,0
6,6843.238770,6844.936035,1
7,6844.936035,6857.415039,1
8,6857.415039,6813.636230,0
9,6813.636230,6807.000977,0


### 10.3. Check Class Balance

In [92]:
forecasting_df[
    "target_direction"
].value_counts(
    normalize=True
)

target_direction
0    0.564103
1    0.435897
Name: proportion, dtype: float64

### 10.4. Remove Last Row

In [93]:
forecasting_df = (
    forecasting_df
    .iloc[:-1]
)

In [94]:
# Validation Missing Values
forecasting_df.isnull().sum()

date                 0
open                 0
high                 0
low                  0
close                0
volume               0
daily_return         1
sma_5                0
sma_10               0
volatility_5         2
news_date           14
avg_sentiment        0
news_count           0
positive_ratio       0
neutral_ratio        0
negative_ratio       0
sentiment_ma_3       0
sentiment_ma_7       0
sentiment_lag_1      0
sentiment_lag_3      0
sentiment_lag_7      0
next_day_close       0
target_direction     0
dtype: int64

In [95]:
# Save data
dio.save_dataframe(
    forecasting_df,
    "../data/interim/forecasting_target.parquet"
)

In [96]:
#Additional validation
forecasting_df[
    [
        "date",
        "close",
        "next_day_close",
        "target_direction"
    ]
].head(10)

,date,close,next_day_close,target_direction
0,2023-02-20,6894.716797,6873.404785,0
1,2023-02-21,6873.404785,6809.967773,0
2,2023-02-22,6809.967773,6839.454102,1
3,2023-02-23,6839.454102,6856.576172,1
4,2023-02-24,6856.576172,6854.776855,0
5,2023-02-27,6854.776855,6843.238770,0
6,2023-02-28,6843.238770,6844.936035,1
7,2023-03-01,6844.936035,6857.415039,1
8,2023-03-02,6857.415039,6813.636230,0
9,2023-03-03,6813.636230,6807.000977,0


## 11. Final Modeling Dataset

Prepare final features and target variables for machine learning.

Tasks:
- Remove non-feature columns
- Remove leakage columns
- Handle remaining missing values
- Save final modeling dataset

### 11.1. Check Missing Values

In [97]:
forecasting_df.isnull().sum()

date                 0
open                 0
high                 0
low                  0
close                0
volume               0
daily_return         1
sma_5                0
sma_10               0
volatility_5         2
news_date           14
avg_sentiment        0
news_count           0
positive_ratio       0
neutral_ratio        0
negative_ratio       0
sentiment_ma_3       0
sentiment_ma_7       0
sentiment_lag_1      0
sentiment_lag_3      0
sentiment_lag_7      0
next_day_close       0
target_direction     0
dtype: int64

### 11.2. Remove Leakage and Unnecessary Columns

In [98]:
forecasting_df = forecasting_df.drop(
    columns=[
        "news_date",
        "next_day_close"
    ]
)

### 11.3. Handle Remaining Missing Values

In [99]:
forecasting_df.isnull().sum()

date                0
open                0
high                0
low                 0
close               0
volume              0
daily_return        1
sma_5               0
sma_10              0
volatility_5        2
avg_sentiment       0
news_count          0
positive_ratio      0
neutral_ratio       0
negative_ratio      0
sentiment_ma_3      0
sentiment_ma_7      0
sentiment_lag_1     0
sentiment_lag_3     0
sentiment_lag_7     0
target_direction    0
dtype: int64

In [100]:
forecasting_df = forecasting_df.dropna()

### 11.4. Final Validation

In [101]:
forecasting_df.isnull().sum()

date                0
open                0
high                0
low                 0
close               0
volume              0
daily_return        0
sma_5               0
sma_10              0
volatility_5        0
avg_sentiment       0
news_count          0
positive_ratio      0
neutral_ratio       0
negative_ratio      0
sentiment_ma_3      0
sentiment_ma_7      0
sentiment_lag_1     0
sentiment_lag_3     0
sentiment_lag_7     0
target_direction    0
dtype: int64

### 11.5. Final Feature Review

Market Features
- open
- high
- low
- close
- volume
- daily_return
- sma_5
- sma_10
- volatility_5

Sentiment Features
- avg_sentiment
- news_count
- positive_ratio
- neutral_ratio
- negative_ratio
- sentiment_ma_3
- sentiment_ma_7
- sentiment_lag_1
- sentiment_lag_3
- sentiment_lag_7

Target
- target_direction

### 11.6. Dataset Shape Check

In [102]:
forecasting_df.shape

(36, 21)

### 11.7. Save Final Dataset

In [103]:
dio.save_dataframe(
    forecasting_df,
    "../data/processed/forecasting_dataset.parquet"
)

In [104]:
daily_sentiment.head()

,news_date,avg_sentiment
0,2023-03-03,0.00
1,2023-03-04,0.50
2,2023-03-05,0.25
3,2023-03-06,0.20
4,2023-03-08,0.00
